# Lenght of stay prediction

@References : Soenksen, L.R., Ma, Y., Zeng, C. et al. Integrated multimodal artificial intelligence framework for healthcare applications. npj Digit. Med. 5, 149 (2022). https://doi.org/10.1038/s41746-022-00689-4

In this notebook, the task is to predict lenght of stay using the CSV embeddings file


## Introduction


The goal of this part of the study is to build models to predict whether or not a patient will be discharged without expiration during the next 48 h as a binary classification problem: discharged alive ≤48 h (1) or otherwise (0). In case of patient death, the class label is set to 0. Each sample in this predictive task corresponds to a single patient-admission EHR time point where an X-ray image was obtained (N = 45,050).


#### Imports

In [1]:
import os
os.chdir('../')

from pandas import read_csv

from src.data import constants
from src.data.dataset import HAIMDataset
from src.evaluation.pycaret_evaluator import PyCaretEvaluator
from src.utils.metric_scores import *

#### Read data from local source



In [2]:
df = read_csv(constants.FILE_DF, nrows=constants.N_DATA)


#### Create a custom dataset for the HAIM experiment


Build the target column for the task at hand, set the dataset specificities:  the ``haim_id`` as a ``global_id``, use all sources for prediction

In [3]:
dataset = HAIMDataset(df,  
                      constants.ALL_PREDICTORS, 
                      constants.ALL_MODALITIES, 
                      constants.LOS, 
                      constants.IMG_ID, 
                      constants.GLOBAL_ID)

#### Set hyper-parameters

In [4]:
# Define the grid oh hyper-parameters for the tuning
grid_hps = {'max_depth': [5, 6, 7, 8],
            'n_estimators': [200, 300],
            'learning_rate': [0.3, 0.1, 0.05],
            }

### Model training and predictions using an XGBClassifier model with GridSearchCV and Hyperparameters optimization


The goal of this section of the notebook is to compute the following metrics:

``ACCURACY_SCORE, BALANCED_ACCURACY_SCORE, SENSITIVITY, SPECIFICITY, AUC, BRIER SCORE, BINARY CROSS-ENTROPY``


The
hyperparameter combinations of individual XGBoost models were
selected within each training loop using a ``fivefold cross-validated
grid search`` on the training set (80%). This XGBoost ``tuning process``
selected the ``maximum depth of the trees (5–8)``, the number of
``estimators (200 or 300)``, and the ``learning rate (0.05, 0.1, 0.3)``
according to the parameter value combination leading to the
highest observed AUROC within the training loop 


As mentioned previously, all XGBoost models were trained ``five times with five different data splits`` to repeat the
experiments and compute average metrics 


```Refer to page 8 of study``` : https://doi.org/10.1038/s41746-022-00689-4

In [5]:
# Initialize the PyCaret Evaluator
evaluator = PyCaretEvaluator(dataset=dataset, target="LengthOfStay", experiment_name="CP_LengthOfStay", filepath="./results/lengthofstay")

# Model training and results evaluation
evaluator.run_experiment(
    train_size=0.8,
    fold=5,
    fold_strategy='stratifiedkfold',
    outer_fold=5,
    outer_strategy='stratifiedkfold',
    session_id=42,
    model='xgboost',
    optimize='AUC',
    custom_grid=grid_hps
)

2024-10-25 22:05:42,189	INFO worker.py:1777 -- Started a local Ray instance. View the dashboard at http://127.0.0.1:8265 


(run_fold pid=249894) Outer fold 1
(run_fold pid=249894) Train indices: [    0     1     3 ... 45046 45047 45049]
(run_fold pid=249894) Test indices: [    2    13    21 ... 45039 45045 45048]


(raylet) Spilled 3869 MiB, 1 objects, write throughput 563 MiB/s. Set RAY_verbose_spill_logs=0 to disable this message.
(raylet) Spilled 7738 MiB, 3 objects, write throughput 564 MiB/s.
(raylet) Spilled 11607 MiB, 5 objects, write throughput 555 MiB/s.


(run_fold pid=249894) Configuring PyCaret for outer fold 1


Processing:  75%|███████▌  | 3/4 [14:44<05:27, 327.11s/it]


...

(run_fold pid=249894)       Accuracy     AUC  Recall   Prec.      F1   Kappa     MCC
(run_fold pid=249894) Fold                                                          
(run_fold pid=249894) 0       0.9584  0.9550  0.6119  0.8539  0.7129  0.6911  0.7023
(run_fold pid=249894) 1       0.9570  0.9611  0.6057  0.8405  0.7041  0.6815  0.6922
(run_fold pid=249894) 2       0.9539  0.9552  0.5617  0.8374  0.6724  0.6486  0.6636
(run_fold pid=249894) 3       0.9575  0.9547  0.5885  0.8640  0.7001  0.6781  0.6927
(run_fold pid=249894) 4       0.9568  0.9591  0.5905  0.8516  0.6974  0.6750  0.6882
(run_fold pid=249894) Mean    0.9567  0.9570  0.5917  0.8495  0.6974  0.6749  0.6878
(run_fold pid=249894) Std     0.0015  0.0026  0.0174  0.0096  0.0135  0.0142  0.0129
(run_fold pid=249894) Tuning hyperparameters for model xgboost with custom grid using grid search


(run_fold pid=249894) Transformation Pipeline and Model Successfully Saved
(run_fold pid=249894)                        Model  Accuracy     AUC  ...      F1   Kappa     MCC
(run_fold pid=249894) 0  Extreme Gradient Boosting    0.9634  0.9684  ...  0.7446  0.7255  0.7389
(run_fold pid=249894) 
(run_fold pid=249894) [1 rows x 8 columns]
(run_fold pid=249894) Outer fold 5
(run_fold pid=249894) Train indices: [    0     1     2 ... 45046 45048 45049]
(run_fold pid=249894) Test indices: [    5     7    14 ... 45040 45043 45047]
(run_fold pid=249894) Configuring PyCaret for outer fold 5


Processing:  75%|███████▌  | 3/4 [14:00<05:10, 310.99s/it]


(run_fold pid=249894)       Accuracy     AUC  Recall   Prec.      F1   Kappa     MCC
(run_fold pid=249894) Fold                                                          
(run_fold pid=249894) 0       0.9553  0.9513  0.5934  0.8281  0.6914  0.6680  0.6788
(run_fold pid=249894) 1       0.9594  0.9600  0.6119  0.8688  0.7181  0.6969  0.7094
(run_fold pid=249894) 2       0.9546  0.9543  0.5658  0.8436  0.6773  0.6539  0.6690
(run_fold pid=249894) 3       0.9572  0.9451  0.5823  0.8654  0.6962  0.6741  0.6894
(run_fold pid=249894) 4       0.9592  0.9611  0.6214  0.8555  0.7199  0.6985  0.7089
(run_fold pid=249894) Mean    0.9571  0.9544  0.5950  0.8523  0.7006  0.6783  0.6911
(run_fold pid=249894) Std     0.0020  0.0059  0.0200  0.0150  0.0163  0.0172  0.0161
(run_fold pid=249894) Tuning hyperparameters for model xgboost with custom grid using grid search


(run_fold pid=249894) Transformation Pipeline and Model Successfully Saved
(run_fold pid=249894)                        Model  Accuracy     AUC  ...      F1   Kappa     MCC
(run_fold pid=249894) 0  Extreme Gradient Boosting    0.9664  0.9693  ...  0.7678  0.7502  0.7623
(run_fold pid=249894) 
(run_fold pid=249894) [1 rows x 8 columns]
Final metrics table:
     Metric     Mean   Std Dev
0  Accuracy  0.95774  0.000913
1       AUC  0.95950  0.003070
2    Recall  0.58768  0.023370
3     Prec.  0.87118  0.024657
4        F1  0.70128  0.010241
5     Kappa  0.67952  0.010264
6       MCC  0.69510  0.007584
Best hyperparameters across all folds: objective                  binary:logistic
base_score                             NaN
booster                             gbtree
callbacks                              NaN
colsample_bylevel                      NaN
colsample_bynode                       NaN
colsample_bytree                       NaN
device                                 cpu
early_stopp